In [ ]:
# take the CSV daily means file, pivot and drop stuff, put into the NEW data csv file

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('../data/Daily_Means_Through_2022.csv')

In [ ]:
print(df.dtypes)

# date to datetime (though we change it again after so idk if this is even smart)
df['date'] = pd.to_datetime(df['date'])
df_pivot = df.pivot_table(index=['date', 'site', 'depth'], columns='parameter', values='measure', aggfunc='mean')
df_pivot.reset_index(inplace=True)
df_pivot['date'] = pd.to_datetime(df_pivot['date'])
df_pivot

In [ ]:
# dropping GB bc too shallow BUT NOT PD because PD IS YEAR ROUND
df_pivot = df_pivot[~df_pivot['site'].str.contains('GB')]
# dropping because many missing values??
df_pivot = df_pivot[~df_pivot['site'].str.contains('CR|TR')]

# drop UB bc we want stratification BUT NOT GD because it is year round
df_pivot = df_pivot[~df_pivot['site'].str.contains('UB')]

# drop HW because it only has surface data and we want stratification
df_pivot = df_pivot[~df_pivot['site'].str.contains('HW')]

# drop BR if depth is mid (not enough data, not really a way to apply this??)
df_pivot = df_pivot[~(df_pivot['site'] == 'BR') | (df_pivot['depth'] != 'mid')]

df_pivot["site"].unique()

In [ ]:
# drop turbidity
df_pivot.drop(columns=['Turb_NTU'], inplace=True)

# # drop Chl a
# df_pivot.drop(columns=['Chl_ug/L'], inplace=True)

# drop DO_pct KEEP THIS FS BECAUSE OTHERWISE IT GIVES IT AWAY
df_pivot.drop(columns=['DO_pct'], inplace=True)

In [ ]:
#impute with mean of each site


# FOR RIGHT NOW NO IMPUTATION
# for site in df_pivot['site'].unique():
#     for depth in df_pivot['depth'].unique():
#         # get the mean value for each parameter at the current site and depth
#         site_depth_df = df_pivot[(df_pivot['site'] == site) & (df_pivot['depth'] == depth)]
#         for column in site_depth_df.columns[3:]:
#             mean_value = site_depth_df[column].mean()
#             # fill null values with the mean value
#             df_pivot.loc[(df_pivot['site'] == site) & (df_pivot['depth'] == depth) & (df_pivot[column].isnull()), column] = mean_value
            
# # check null counts again
# null_counts = df_pivot.isnull().sum()
# print(null_counts)
# print(len(df_pivot))
# print(df_pivot)



In [ ]:
# find station and depth where density is missing
missing_density = df_pivot[df_pivot['Density_g/cm3'].isnull()]
print(missing_density[['site', 'depth']].drop_duplicates())

In [ ]:
# missing_density = df_pivot[df_pivot['Depth_m'].isnull()]
# print(missing_density[['site', 'depth']].drop_duplicates())

In [ ]:
# check null counts again
null_counts = df_pivot.isnull().sum()
print(null_counts)
print(len(df_pivot))


# not too shabby! DO there for like 98% of the data

In [ ]:
# is there a top depth for every bottom depth?

df_pivot

In [ ]:
# get number of depths for each site
print(df_pivot.value_counts(subset=['site', 'depth']))

In [ ]:
df_pivot

In [ ]:
df_new = df_pivot.pivot_table(index=['date', 'site'], 
                          columns='depth', 
                          values=['DO_mg/L', 'Density_g/cm3', 'Depth_m', 'Salinity_ppt', 'Temp_C', 'pH', 'Chl_ug/L'],)
df_new

In [ ]:
df_pivot.columns

In [ ]:
# Flatten the multi-index columns
df_new.columns = [f"{pos}_{col}" for col, pos in df_new.columns]

# Reset index to make it a regular dataframe
df_new = df_new.reset_index()

# Display result
print(df_new.head())

In [ ]:
df_new

In [ ]:
df_new = df_new.sort_values(by=['site'])

In [ ]:
df_new

In [ ]:
# null counts
print(df_new.isnull().sum())


In [ ]:
# put into a csv file
df_new.to_csv('../data/Daily_Means_Through_2022_cleaned_NEW.csv', index=False)

In [ ]:
df_new.columns